# SQL Server Smoke Test — VDI / Windows

**Goal:** prove this VDI can connect to `RSD-RBSQLDEV` / `BI_SANDBOX` from Python.

**Time:** 2 minutes.

**What to do:** click into the first code cell below and press **Shift + Enter** to run each cell, one at a time. Don't skip cells. If a cell prints a red error, stop and screenshot it.

**If everything passes:** you'll see a row count and a 5-row preview at the bottom. Send that screenshot to Prosper.


## Step 1 — Install the two packages we need

Run this once. If it says they're already installed, that's fine.

In [ ]:
%pip install pyodbc pandas

## Step 2 — Restart the kernel

**IMPORTANT:** after Step 1 finishes, go to the top menu: **Kernel → Restart**.
Then come back and run Step 3.

(If you skip this, the import below will say 'No module named pyodbc' even though you just installed it.)

## Step 3 — Check the ODBC driver is installed on this VDI

In [ ]:
import pyodbc

drivers = [d for d in pyodbc.drivers() if 'SQL Server' in d]
print('SQL Server ODBC drivers found on this VDI:')
for d in drivers:
    print('  -', d)

if not drivers:
    print('\nNONE found. Stop and send this output to Prosper.')
else:
    # Prefer 18 if available, else 17, else whatever's there
    SQL_DRIVER = next((d for d in drivers if '18' in d),
                     next((d for d in drivers if '17' in d), drivers[-1]))
    print(f'\nWill use: {SQL_DRIVER}')

## Step 4 — Connect to the SQL Server

Uses your Windows login automatically (Trusted_Connection). You should NOT need to type a password.

In [ ]:
SQL_SERVER   = 'RSD-RBSQLDEV'
SQL_PORT     = 1433
SQL_DATABASE = 'BI_SANDBOX'

conn_str = (
    f'DRIVER={{{SQL_DRIVER}}};'
    f'SERVER={SQL_SERVER},{SQL_PORT};'
    f'DATABASE={SQL_DATABASE};'
    'Encrypt=yes;'
    'TrustServerCertificate=yes;'
    'Trusted_Connection=yes;'
)

print(f'Connecting to {SQL_SERVER}:{SQL_PORT}/{SQL_DATABASE} as your Windows account...')
conn = pyodbc.connect(conn_str, timeout=15)
print('  CONNECTED')

cur = conn.cursor()
cur.execute('SELECT SYSTEM_USER, SUSER_SNAME(), DB_NAME(), @@SERVERNAME')
user, login, db, server = cur.fetchone()
print(f'  Server : {server}')
print(f'  Database: {db}')
print(f'  Connected as: {user}  (login: {login})')

## Step 5 — List the tables you can see

In [ ]:
import pandas as pd

tables = pd.read_sql("""
    SELECT TABLE_SCHEMA, TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME DESC
""", conn)

print(f'Tables visible in {SQL_DATABASE}: {len(tables)}')
tables.head(20)

## Step 6 — Pull a few rows from one of the BASE tables

Confirms we can actually read data, not just metadata.

In [ ]:
# Pick the most recent BASE<YYYYMM> table that exists
base_tables = tables[tables['TABLE_NAME'].str.startswith('BASE')].sort_values('TABLE_NAME', ascending=False)

if base_tables.empty:
    print('No BASE* table found yet. Step 5 should still have listed something.')
    print('Stop here and send Prosper the output of Step 5.')
else:
    target = base_tables.iloc[0]
    qualified = f"[{target['TABLE_SCHEMA']}].[{target['TABLE_NAME']}]"
    print(f'Reading from {qualified} ...\n')

    count_df = pd.read_sql(f'SELECT COUNT(*) AS n FROM {qualified}', conn)
    print(f"Row count: {int(count_df['n'].iloc[0]):,}")

    preview = pd.read_sql(f'SELECT TOP 5 * FROM {qualified}', conn)
    print(f"\nFirst 5 rows ({len(preview.columns)} columns):")
    preview

## Done

If you got a row count and a preview table above, **you're connected**. Screenshot Steps 4, 5, and 6 and send to Prosper. We'll do the next step (the full upload pipeline) after that.

Close the connection:

In [ ]:
conn.close()
print('Connection closed.')